# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
import duckdb
from google.colab import userdata


HF_TOKEN = userdata.get("FLYRANK_AI")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
);
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

result = con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

result.show()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘



## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the daily search and engagement performance of a single content page for a single client on one report date.

The analysis uses the warehouse data for March 2026 (month='2026-03'). This mid-panel month is used to avoid developing logic on the final month (June 2026), which is reserved as a sealed test period.

In [17]:
result = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03';
""")

result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬────────────┬────────────┐
│ row_count │ start_date │  end_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [18]:
import pandas as pd

field_contract = pd.DataFrame({
    "Category": [
        "Feature",
        "Feature",
        "Feature",
        "Feature",
        "Feature",
        "Label / Proxy",
        "Context",
        "Context",
        "Context",
        "Excluded"
    ],
    "Field": [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_engaged_sessions",
        "scroll_events",
        "Refresh Priority (ranking target)",
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "Future metrics / label-derived columns"
    ],
    "Reason": [
        "Historical search visibility available before decision.",
        "Historical search clicks available before decision.",
        "Historical average search position.",
        "Historical engagement signal.",
        "Historical user interaction signal.",
        "Target used for prioritizing pages for refresh.",
        "Identifies the client.",
        "Identifies the content page.",
        "Identifies the observation date.",
        "Excluded to prevent data leakage."
    ]
})

field_contract

,Category,Field,Reason
0,Feature,gsc_impressions,Historical search visibility available before ...
1,Feature,gsc_clicks,Historical search clicks available before deci...
2,Feature,gsc_avg_position,Historical average search position.
3,Feature,ga4_engaged_sessions,Historical engagement signal.
4,Feature,scroll_events,Historical user interaction signal.
5,Label / Proxy,Refresh Priority (ranking target),Target used for prioritizing pages for refresh.
6,Context,client_hash_id,Identifies the client.
7,Context,content_hash_id,Identifies the content page.
8,Context,report_date,Identifies the observation date.
9,Excluded,Future metrics / label-derived columns,Excluded to prevent data leakage.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [19]:
result = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS pages,
    COUNT(DISTINCT report_date) AS dates
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03';
""")

result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────┬────────┬───────┐
│ total_rows │ clients │ pages  │ dates │
│   int64    │  int64  │ int64  │ int64 │
├────────────┼─────────┼────────┼───────┤
│    9841378 │      55 │ 331437 │    31 │
└────────────┴─────────┴────────┴───────┘



### Query 1 — Verify the grain

The expected grain is one row per client, content page, and report date. This query checks that the row count matches the number of distinct `(client_hash_id, content_hash_id, report_date)` combinations.

In [20]:
result = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS cnt
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03'
GROUP BY 1,2,3
HAVING COUNT(*) > 1;
""")

result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────┐
│ client_hash_id │ content_hash_id │ report_date │  cnt  │
│    varchar     │     varchar     │    date     │ int64 │
├────────────────┴─────────────────┴─────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘



No rows are returned. Hence, Grain is verified

### Query 2 — Verify row count and date span

This query confirms how many rows are present in the March 2026 slice and the minimum and maximum report dates covered by the slice.

In [21]:
result = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03';
""")

result.show()

┌───────────┬────────────┬────────────┐
│ row_count │ start_date │  end_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



This confirms the dataset slice covers March 2026.

The March 2026 slice contains 9841378 of rows and covers the expected date range.

### Query 3 — Verify data availability

GA4 availability is checked explicitly with `IS TRUE`, as required. This shows how many March 2026 rows have GA4 data available.

In [22]:
result = con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03'
  AND ga4_data_available IS TRUE;
""")

result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│         413966 │
└────────────────┘



The query confirms the number of March 2026 rows for which GA4 data is explicitly available.

## **FIVE FEATURES:**

In [23]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')

""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [24]:
features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_engaged_sessions,
    scroll_events
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03'
  AND ga4_data_available IS TRUE
LIMIT 1000;
""")

features.df().head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,scroll_events
0,client_65de48885f4ef01b,content_09be8cc7fcb222af,2026-03-01,0,0,NaN,0,0
1,client_65de48885f4ef01b,content_851afac9fe13612e,2026-03-01,0,0,NaN,0,0
2,client_65de48885f4ef01b,content_cee6c6fc8c51af14,2026-03-01,0,0,NaN,0,0
3,client_65de48885f4ef01b,content_5e120e972f11f833,2026-03-01,0,0,NaN,0,0
4,client_65de48885f4ef01b,content_16a7291bb6ecaebe,2026-03-01,0,0,NaN,0,0


## Five Selected Features

### Feature 1: `gsc_impressions`
**Available when?**  
Knowable at the decision moment because search impressions are historical observations already available before deciding whether a page should be refreshed.

---

### Feature 2: `gsc_clicks`
**Available when?**  
Knowable at the decision moment because historical click data has already been collected before the refresh decision.

---

### Feature 3: `gsc_avg_position`
**Available when?**  
Knowable at the decision moment because the average search position is computed from historical Google Search Console data available before making the decision.

---

### Feature 4: `ga4_engaged_sessions`
**Available when?**  
Knowable at the decision moment because user engagement metrics are collected before deciding whether a page should be refreshed.

---

### Feature 5: `scroll_events`
**Available when?**  
Knowable at the decision moment because scroll events represent historical user interactions with the page and are observed before the refresh decision.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limitation

This warehouse snapshot is an **unbalanced panel**, meaning different clients have different amounts of historical data. Some rows contain only Google Search Console data because GA4 tracking started later.

Additionally, this dataset is observational. It can identify pages that appear to be good candidates for content refresh, but it cannot prove that refreshing a page will cause future performance improvements. Any recommendations should therefore be treated as decision support rather than causal evidence.

## 5. Deliberate Leakage Experiment

To demonstrate target leakage, I create a simple proxy label indicating whether a page-day received at least one Google Search Console click:

`has_click = 1` when `gsc_clicks > 0`, otherwise `0`.

I first train a quick model using only historical observable features. I then deliberately add a feature derived directly from the label. This leaked feature should make the score unrealistically close to perfect because it gives the model the answer directly.

After demonstrating the inflated score, I remove the leaked feature and report the honest score. The leaked feature is not retained in the feature set.

In [25]:
leak_data = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_avg_position,
    ga4_engaged_sessions,
    scroll_events,
    CASE
        WHEN gsc_clicks > 0 THEN 1
        ELSE 0
    END AS has_click
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND gsc_impressions IS NOT NULL
  AND gsc_clicks IS NOT NULL
  AND gsc_avg_position IS NOT NULL
LIMIT 20000;
""").df()

leak_data.head()

,gsc_impressions,gsc_avg_position,ga4_engaged_sessions,scroll_events,has_click
0,20,3.350000,<NA>,<NA>,0
1,1,0.000000,<NA>,<NA>,0
2,125,4.928000,<NA>,<NA>,1
3,7,4.000000,<NA>,<NA>,0
4,11,2.272727,<NA>,<NA>,0


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

features = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_engaged_sessions",
    "scroll_events"
]

X = leak_data[features]
y = leak_data["has_click"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    LogisticRegression(max_iter=1000)
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict_proba(X_test)[:, 1]

honest_auc = roc_auc_score(y_test, honest_pred)

print(f"Honest ROC AUC: {honest_auc:.3f}")

Honest ROC AUC: 0.878


In [27]:
leak_data["label_derived_feature"] = leak_data["has_click"]

leaked_features = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_engaged_sessions",
    "scroll_events",
    "label_derived_feature"
]

X_leaked = leak_data[leaked_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_leaked,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

leaked_model = make_pipeline(
    SimpleImputer(strategy="median"),
    LogisticRegression(max_iter=1000)
)

leaked_model.fit(X_train, y_train)

leaked_pred = leaked_model.predict_proba(X_test)[:, 1]

leaked_auc = roc_auc_score(y_test, leaked_pred)

print(f"Leaked ROC AUC: {leaked_auc:.3f}")

Leaked ROC AUC: 1.000


In [28]:
leak_data = leak_data.drop(columns=["label_derived_feature"])
print("label_derived_feature" in leak_data.columns)

False


In [29]:
comparison = pd.DataFrame({
    "Experiment": [
        "Honest features",
        "With deliberate leakage"
    ],
    "ROC_AUC": [
        honest_auc,
        leaked_auc
    ]
})

comparison

,Experiment,ROC_AUC
0,Honest features,0.877879
1,With deliberate leakage,1.000000


### Leakage Finding

The model score increased sharply after adding `label_derived_feature` because that feature was created directly from the target (`has_click`). The model therefore received the answer it was supposed to predict.

This is data leakage: information derived from the label is allowed into the feature set, making evaluation look unrealistically strong.

After removing `label_derived_feature`, the honest ROC AUC is retained as the valid result. The leaked feature is not included in the final feature set.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.